# Load Counts, Doublet Integration, and Cell Cycle Scoring for snRNA-seq

Loads per-sample CellBender-denoised 10x count matrices, merges them into a
single AnnData object, integrates previously computed Scrublet doublet
scores, and scores cell-cycle phase.

## Setup

In [ ]:
import os

# Set cache directories before importing numba/matplotlib-dependent packages
os.environ["NUMBA_CACHE_DIR"] = "/path/to/tmp"
os.environ["MPLCONFIGDIR"] = "/path/to/tmp"

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sc.settings.verbosity = 1  # errors (0), warnings (1), info (2), hints (3)

## Config

Set metadata, input/output paths, and thresholds.

In [ ]:
meta_path = "/path/to/metadata/meta_snRNAseq_batch1.csv"
data_dir = "/path/to/cellranger/fastqs/"
scrublet_scores_dir = "/path/to/results/scrublet-scores-cellbender/"
cell_cycle_genes_path = "/path/to/regev_lab_cell_cycle_genes.txt"
output_path = "/path/to/output/S1_counts_adata.h5ad"

doublet_pval_threshold = 0.01

## Load sample metadata

In [ ]:
meta = pd.read_csv(meta_path, index_col=0)
meta.head()

## Load per-sample count matrices

In [ ]:
def load_counts(meta, data_dir):
    """Load CellBender-denoised 10x count matrices for each sample in meta,
    attach per-sample metadata, and compute mito fraction / total counts."""
    holder = []
    for sample in meta.index:
        print(sample)
        adata_sample = sc.read_10x_h5(
            os.path.join(data_dir, sample, f"{sample}_sn10x3GEX", "outs", "cellbender_output_filtered.h5")
        )
        print("Original number of cells: {:d}".format(adata_sample.n_obs))

        adata_sample.var_names_make_unique()
        adata_sample.obs_names = [f"{sample}_" + bc.split("-")[0] for bc in adata_sample.obs_names]

        # attach sample-level metadata
        adata_sample.obs["sample"] = sample
        for col in meta.columns:
            adata_sample.obs[col] = meta[col][sample]

        # mito fraction and total counts
        mito_genes = [name for name in adata_sample.var_names if name.startswith("MT-")]
        adata_sample.obs["percent_mito"] = (
            np.sum(adata_sample[:, mito_genes].X, axis=1).A1 / np.sum(adata_sample.X, axis=1).A1
        )
        adata_sample.obs["n_counts"] = adata_sample.X.sum(axis=1).A1

        print("Total number of cells: {:d}".format(adata_sample.n_obs))
        print("Total number of genes: {:d}".format(adata_sample.n_vars))

        holder.append(adata_sample)
    return holder

In [ ]:
holder = load_counts(meta, data_dir)

# merge samples, taking the union (outer join) of genes
adata = holder[0].concatenate(holder[1:], join="outer", index_unique=None)
adata.X = adata.X.tocsr()
adata

In [ ]:
# keep n_genes in obs metadata (no cells are actually dropped at min_genes=1)
sc.pp.filter_cells(adata, min_genes=1)
print("Total number of cells: {:d}".format(adata.n_obs))

## QC overview

Visualize basic QC metrics per sample before proceeding.

In [ ]:
sc.pl.violin(adata, ["n_genes", "n_counts", "percent_mito"], groupby="sample", rotation=90)

## Integrate Scrublet doublet scores

Requires per-sample doublet score CSVs already generated (see `S00_doubletsDetection.ipynb`).

In [ ]:
scorenames = ["scrublet_score", "scrublet_cluster_score", "zscore", "bh_pval", "bonf_pval"]

scrdf = []
for sample in meta.index:
    scrdf.append(
        pd.read_csv(os.path.join(scrublet_scores_dir, f"{sample}.csv"), header=0, index_col=0)
    )
scrdf = pd.concat(scrdf)
scrdf.index = [i.replace("-1", "") for i in scrdf.index]

for score in scorenames:
    adata.obs[score] = scrdf[score]

adata.obs["is_doublet"] = adata.obs["bonf_pval"] < doublet_pval_threshold
print("Total number of doublets: {:d}".format(len(adata.obs.loc[adata.obs["is_doublet"] == True])))

## Score cell cycle phase

In [ ]:
cell_cycle_genes = [x.strip() for x in open(cell_cycle_genes_path)]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]
s_genes = [x for x in s_genes if x in adata.var_names]
g2m_genes = [x for x in g2m_genes if x in adata.var_names]

sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
# optional: regress out cell-cycle effect
# sc.pp.regress_out(adata, ["S_score", "G2M_score"])

## Save output

In [ ]:
adata.layers["raw_counts"] = adata.X.copy()
adata.write(output_path)
adata